# 테이블 생성/데이터 입력 노트북

이 노트북은 다음 순서로 사용합니다.
1. `TABLE_NAME`, `SCHEMA`를 직접 작성
2. 테이블 생성 셀 실행
3. `ROWS`에 데이터 작성
4. 데이터 입력/조회 셀 실행


# 문항 데이터

In [1]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

# DB 경로: 프로젝트 루트 app.db (필요 시 이 줄만 수정)
DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\app.db")
# DB_PATH = Path(r"C:\Users\kmj\project\__project_original\2.0-modular\app.db")
TABLE_NAME = 'parent_item'  # 원하는 테이블 이름으로 변경

# type: INTEGER(정수) / REAL(실수) / TEXT / BLOB(바이너리) 중 하나 사용
# constraints 예시: 'PRIMARY KEY AUTOINCREMENT', 'NOT NULL', 'DEFAULT 0'
SCHEMA = [
    {'name': 'test_id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 검사명
    {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 세부검사 조건
    {'name': 'item_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 검사 문항
    {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'} # 검사 메타데이터
]
PRIMARY_KEY = ['test_id', 'sub_test_json'] # 복합 pk 설정

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)

설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\app.db
TABLE_NAME = parent_item
SCHEMA = [{'name': 'test_id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'item_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}]
PRIMARY_KEY = ['test_id', 'sub_test_json']


In [4]:
# 2) 테이블 생성
column_defs = []
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').strip()
    part = f'"{c_name}" {c_type}'
    if c_constraints:
        part += f' {c_constraints}'
    column_defs.append(part)

if PRIMARY_KEY:
    pk_cols = ', '.join([f'"{c}"' for c in PRIMARY_KEY])
    column_defs.append(f'PRIMARY KEY ({pk_cols})')

create_sql = f'CREATE TABLE IF NOT EXISTS "{TABLE_NAME}" (' + ', '.join(column_defs) + ')'
print('실행 SQL:', create_sql)

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(create_sql)
    conn.commit()
    print(f'테이블 생성 완료: {TABLE_NAME}')
finally:
    conn.close()

실행 SQL: CREATE TABLE IF NOT EXISTS "parent_item" ("test_id" TEXT NOT NULL, "sub_test_json" TEXT NOT NULL, "item_json" TEXT NOT NULL, "meta_json" TEXT NOT NULL, PRIMARY KEY ("test_id", "sub_test_json"))
테이블 생성 완료: parent_item


## 업서트

In [2]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'GOLDEN',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [18, 0, 0],
                'end_exclusive': [99, 99, 99],
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            "1": "처음 만난 사람들과 있게 되면, 나는",
            "2": "나는 주로",
            "3": "휴가 중에 내가 더 많은 시간을 쓰고 싶은 것은",
            "4": "게임이나 스포츠를 처음 시작할 때, 나는",
            "5": "나만의 정원을 꾸민다면, 내가 좋아하는 정원은",
            "6": "모임에서 떠나려 할 때, 나는",
            "7": "프로젝트를 시작하기 전에, 나는",
            "8": "어떤 사람이 감정적일 경우,",
            "9": "다른 사람에게 나에 대하여 알려주고 싶은 점은",
            "10": "일과 관련하여 대부분의 사람들은 나를",
            "11": "내가 잘 알고 있는 분야는",
            "12": "지출 계획을 세울 때, 나는",
            "13": "나는 일반적으로",
            "14": "연설을 요청받는다면, 나는",
            "15": "자동차나 버스에서 내 뒷자리에 있는 사람들이 대화할 때, 나는 대부분",
            "16": "예상치 못한 일이 발생하여 내 계획에 영향을 줄 때, 나는",
            "17": "삶이란",
            "18": "일상에서의 귀찮은 일들과 걸림돌은, 나에게",
            "19": "길을 함께 가던 일행이 내가 가자고 한 방향이 틀리다고 말하면,",
            "20": "페인트 칠처럼 사전 준비 작업이 많은 일을 할 때, 나는",
            "21": "많은 사람들이 주위에 있을 때, 나는",
            "22": "사교 모임에서, 나는",
            "23": "사람들이 나를 어떻게 생각하든지, 나는 정말",
            "24": "사교 모임에서, 나는 주로",
            "25": "나는 전에 가본 적 없는 도시를 자동차로 여행할 때",
            "26": "내가 집을 짓는다면,",
            "27": "계획을 세울 때, 나는",
            "28": "대부분의 대화에서, 나는",
            "29": "내가 주로 좋아하는 사람들은",
            "30": "비즈니스 차트나 그래프를 볼 때, 나는",
            "31": "휴가 갈 때, 나는",
            "32": "나에 대해 좀 더 정확히 말한다면,",
            "33": "새로운 일을 시작하거나 새로운 곳으로 이사했을 때, 나는 주로 새로운 친구를",
            "34": "다른 사람의 이야기를 들을 때, 나는",
            "35": "만약 누군가가 나에 대해 험담을 하면, 나는",
            "36": "지난 2년 동안, 나는",
            "37": "원하지 않는 메일을 많이 받을 때, 나는",
            "38": "중요한 사안에 대해 내 입장을 결정할 때, 나는 가장 먼저",
            "39": "나는 주로",
            "40": "반대 의견에 대해,나는",
            "41": "나는 일반적으로 주말 저녁에",
            "42": "휴일 계획을 세우거나 친목모임을 계획할 때, 나는",
            "43": "고지서를 받았을 때, 나는",
            "44": "내가 선호하는 업무환경은",
            "45": "나는 상당히",
            "46": "나는 많은 사람들과 같이 있을 때,",
            "47": "내가 잘 모르는 사람들과 대화를 해야 할 때, 나는",
            "48": "나는",
            "49": "슬픈 영화를 볼 때, 나는",
            "50": "새로운 물건을 조립할 때, 나는",
            "51": "고가의 상품을 산 후, 나는",
            "52": "나는 종종",
            "53": "내가 주로 읽고 싶거나 듣고 싶은 것들은",
            "54": "내가 잘 알고 있는 두 사람 간의 논쟁을 들을 때, 나는",
            "55": "세부적인 것에 집중할 때, 나는",
            "56": "지난 한 해 동안, 나는 자주",
            "57": "내가 살고 싶은 곳은",
            "58": "내가 더 존경하고 동경하는 사람은",
            "59": "나는 의사결정을 할 때,",
            "60": "약속이 있을 때, 나는",
            "61": "긴 여행을 떠나기 전에, 나는",
            "62": "나는 근본적으로 내 인생이 어떻게 될지에 대하여",
            "63": "문제를 해결할 때, 나는",
            "64": "진정한 나의 모습은",
            "65": "내가 좋아하는 강사나 워크숍 진행자는",
            "66": "관심 있는 스포츠를 관람할 때, 나는",
            "67": "내가 기분 좋다고 느낄 때는",
            "68": "어떤 일을 반복해서 할 때, 나는",
            "69": "나의 미래는",
            "70": "내가 생각하기에",
            "71": "약속시간을 지키는 것은, 나에게",
            "72": "나에게 더 중요한 것은",
            "73": "결정을 할 때, 대체적으로 나는",
            "74": "내가 추구하는 것은",
            "75": "안전하며 변함없는 안정된 환경이다.",
            "76": "나의 분석 능력을 활용할 기회이다.",
            "77": "규칙이 강조되는 세상보다 서로 배려하는 세상을 만드는 것이다.",
            "78": "내가 새로운 프로젝트를 맡으려는 이유는 나에게 주어진 업무이기 때문이다.",
            "79": "유명해질 수 있기 때문이다.",
            "80": "흥미롭기 때문이다.",
            "81": "도전적이고 경제적 보상이 크기 때문이다.",
            "82": "내게 이상적인 직업이란 어느정도 안정적인 수입이 있어야 한다.",
            "83": "내가 정말 좋아하고 아끼는 동료들이 있어야 한다.",
            "84": "지적인 자극을 주어야 한다.",
            "85": "도전적이고 흥미로워야 한다.",
            "86": "팀에 기여할 수 있는 장점 중에서, 내가 특히 잘하는 것은, 100% 최선을 다하며, 결정된 일을 실행하는 것이다.",
            "87": "새로운 아이디어와 개념들을 제시하는 것이다.",
            "88": "일에 중심을 두기 보다는 다른 사람들의 기분, 신념, 요구를 배려하는 것이다.",
            "89": "사람들에게 동기를 부여하는 것이다.",
            "90": "어떤 것이 진실이라고 생각되는 것은, 내가 진실이라고 믿는 것과 부합하는 경우이다.",
            "91": "그것이 과학적인 근거가 있는 경우이다.",
            "92": "실제라고 증명되는 경우이다.",
            "93": "나의 자질 중에서 내가 칭찬하고 싶은 것은 의사결정을 할 때 원칙대로 일하기보다는 다른 사람의 입장을 잘 고려하는 것이다.",
            "94": "신속하고 신뢰가 높다는 것이다.",
            "95": "임기응변에 뛰어난 것이다.",
            "96": "규칙을 잘 따르는 것이다.",
            "97": "내가 특히 잘하는 것은 사람을 객관적으로 대하기보다는, 그 사람의 생각과 감정을 이해하는 것이다.",
            "98": "위기를 해결하는 것이다.",
            "99": "적절한 순서에 맞게 일을 처리하는 것이다.",
            "100": "장기적인 비전이나 계획을 제시하는 것이다.",
            "101": "내 성향을 말하자면, 혼자서 시간 보내는 것을 좋아한다.",
            "102": "많은 친구를 사귀기보다 속마음을 얘기할 수 있는 단짝 친구를 사귀는 편이다.",
            "103": "한두 명의 친한 친구를 사귀는 편이다.",
            "104": "친구나 지인들과 폭넓게 네트워크를 형성하는 편이다.",
            "105": "사람들은 나를 분석적인 사람이라고 생각한다.",
            "106": "뭔가 새로운 비전을 제시하는 사람이라고 생각한다.",
            "107": "책임감 있는 사람이라고 생각한다.",
            "108": "심사숙고하기 보다는 모험을 감수하는 사람이라고 생각한다.",
            "109": "내가 자랑스러워하는 나의 모습은 수완이 좋은 협상가이다.",
            "110": "전략적인 사고자이다.",
            "111": "전술적인 문제해결자이다.",
            "112": "세부적인 것을 잘 다루는 숙련자이다.",
            "113": "팀 프로젝트에서 의사결정을 할 때, 내가 먼저 하는 일은 무엇을 성취해야 할지, 무엇이 부족한지 알아보는 것이다.",
            "114": "일의 특징을 파악하기 보다는, 그 일이 주변 사람에게 어떤 영향을 미칠지 생각하는 것이다.",
            "115": "업무 목록을 정하고 일정을 세우는 것이다.",
            "116": "프로젝트의 주요 찬성자들과 반대자들을 만나는 것이다.",
            "117": "친구들과 여름에 야외수영장에 갔을 때, 내가 하고 싶은 것은 편한 의자에서 쉬는 것이다.",
            "118": "물놀이를 하는 것이다.",
            "119": "음식과 음료를 준비하는 것이다.",
            "120": "즐거운 대화를 나누는 것이다.",
            "121": "내가 선호하는 평가 방식은 ‘객관식 시험’이다.",
            "122": "‘구두 발표’이다.",
            "123": "‘보고서’이다.",
            "124": "일을 하면서 내가 기분이 좋을 때는 정해진 규칙과 절차를 어떻게 따라야 할 지 알고 있을 때이다.",
            "125": "다른 사람들과 조화를 이루면서 의사를 결정할 때이다.",
            "126": "새로운 것을 소개하고 그것이 수용될 때이다.",
            "127": "다양한 프로젝트들에 참여할 때이다.",
            "128": "내가 효과적으로 일할 수 있는 때는 주어진 목표가 명확하고 측정 가능하며, 그 목표를 달성하는 방법을 알려줄 때이다.",
            "129": "이전에 해결하지 못했던 문제를 포함한 새로운 업무를 받았을 때이다.",
            "130": "도전적인 일이 주어지고 그 일을 내 방식대로 해결할 때다.",
            "131": "다른 사람들이 최고의 성과를 낼 수 있도록 그들을 북돋울 때이다.",
            "132": "",
            "133": "",
            "134": "",
            "135": "",
            "136": "",
            "137": "",
            "138": "",
            "139": "",
            "140": "",
            "141": "",
            "142": "",
            "143": "",
            "144": "",
            "145": "",
            "146": "",
            "147": "",
            "148": "",
            "149": "",
            "150": "",
            "151": "",
            "152": "",
            "153": "",
            "154": "",
            "155": "",
            "156": "",
            "157": "",
            "158": "",
            "159": "",
            "160": "",
            "161": "",
            "162": "",
            "163": "",
            "164": "",
            "165": "",
            "166": "",
            "167": "",
            "168": ""
        },
        'meta_json': {'name': 'GOLDEN 성격유형검사 성인용',
                      'item_count': 168
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건


In [6]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(f'SELECT * FROM "{TABLE_NAME}" ORDER BY rowid DESC')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()

총 1건
('STS', '{"age_range": {"as_of_time": "00:00:00", "start_inclusive": [1, 0, 0], "end_exclusive": [3, 0, 0]}, "gender": ["male", "female"]}', '{"1": "주변이 소란스러워도 하던 놀이를 계속한다.", "2": "기분이 나쁠 때보다는 좋을 때가 더 많다.", "3": "목욕할 때 물을 튀기거나 발로 차는 등 많이 움직인다.", "4": "새로운 옷(또는 신발)을 입지 않으려고 한다.", "5": "뜻대로 되지 않으면 쉽게 운다.", "6": "가끔 보는 지인이 집에 오면 낯을 가린다.", "7": "다른 아이가 울면 따라 운다.", "8": "주의력이 요구되는 활동(퍼즐이나 책 등)을 좋아한다.", "9": "도전적인 신체 활동을 좋아한다(예: 높은 곳에 기어오르기).", "10": "항상 미소 짓거나 웃는다.", "11": "컨디션이 좋지 않을 때 지나치게 칭얼거린다.", "12": "낯선 성인을 만났을 때 엄마에게 매달린다.", "13": "잠자고 일어날 때면 짜증을 내거나 운다.", "14": "하루 중 대부분의 시간을 기분 좋게 보낸다.", "15": "놀이터나 키즈카페에서 놀 때 행동이 재빠르다.", "16": "놀이할 때 잘 웃는다.", "17": "익숙하지 않은 상황을 피하려고 한다.", "18": "처음 보는 친구에게 다가가기 어려워한다.", "19": "다른 사람이 다치는 것을 보면 움츠러든다.", "20": "새로운 활동을 시작하는 것을 주저한다.", "21": "사물보다 사람에게 관심이 더 많다.", "22": "처음 먹어보는 음식은 먹지 않으려고 한다.", "23": "잠들기 전에 잠투정이 있다.", "24": "사람들의 외적인 변화에 관심을 보인다(예: 안경, 수염, 헤어스타일, 액세서리 등).", "25": "매일 밖에 나가서 놀자고 한다.", "26": "좋아하는 장난감을 한참을 가지고 논다.", "27": "동화책

## 아동용

In [3]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'GOLDEN',
        'sub_test_json': {
            'school_age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [4, 0, 0], # 4 = 초등학교 4학년
                'end_exclusive': [15, 0, 0], # 15 = 고등학교 졸업
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            "1": "나는 처음 만난 사람들과 있게 되면,",
            "2": "내가 더 활용하는 것은",
            "3": "나는 상대방이 어려운 상황에 처하면,",
            "4": "나는 계획을 세울 때,",
            "5": "나는 어떤 문제가 생겼을 때,",
            "6": "나는 다른 사람보다 말을",
            "7": "내가 좋아하는 과제는",
            "8": "나는 슬픈 영화를 볼 때",
            "9": "나는 일을 할 때",
            "10": "나는",
            "11": "내가 좋아하는 것은",
            "12": "내가 원하는 것은",
            "13": "내가 주로 의존하는 것은",
            "14": "나는 여행하려고 짐을 챙길때",
            "15": "나의 미래는",
            "16": "나는 많은 사람과 같이 있는 상황을",
            "17": "내가 하고 싶은 과제는",
            "18": "나는 친구가 시험을 망치면",
            "19": "나는 결정할 때,",
            "20": "나는",
            "21": "나는 시간이 나면,",
            "22": "나는",
            "23": "나는",
            "24": "나는",
            "25": "나는 다른 사람들이 나를 어떻게 생각할지에 대해",
            "26": "나는 모임에서,",
            "27": "내가 좋아하는 디자인은",
            "28": "나는 다른 사람의 이야기를 들을 때,",
            "29": "나는",
            "30": "나는 새로운 친구를 사귀기가",
            "31": "나는",
            "32": "내가 좋아하는 사람은",
            "33": "나는",
            "34": "나는",
            "35": "나는",
            "36": "나는 먼저",
            "37": "나는 문제 상황에서",
            "38": "나는 많은 사람들과 같이 있을 때,",
            "39": "나는",
            "40": "나는",
            "41": "내가 좋아하는 과제는",
            "42": "나는",
            "43": "나는 숙제를 할 때,",
            "44": "",
            "45": "",
            "46": "",
            "47": "",
            "48": "",
            "49": "",
            "50": "",
            "51": "",
            "52": "",
            "53": "",
            "54": "",
            "55": "",
            "56": "",
            "57": "",
            "58": "",
            "59": "",
            "60": "",
            "61": "",
            "62": "",
            "63": "",
            "64": "",
            "65": "",
            "66": "",
            "67": "",
            "68": "",
            "69": "",
            "70": "",
            "71": "",
            "72": ""
        },
        'meta_json': {'name': 'STS 6요인 기질검사 유아용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건


## 성인용

In [10]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [18, 0, 0],
                'end_exclusive': [100, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            '1': '다른 사람의 속뜻이나 진짜 기분을 쉽게 파악할 수 있다.',
            '2': '기분이 나쁠 때보다는 좋을 때가 많다.',
            '3': '처음 만나는 사람에게 쉽게 다가가기 어렵다.',
            '4': '사소한 일에도 기분이 나빠진다.',
            '5': '별로 힘들이지 않고 하루 종일 활동할 수 있다.',
            '6': '내 삶에 만족한다.',
            '7': '다른 사람의 기분을 잘 알아챈다.',
            '8': '일을 마칠 때까지는 피곤한 줄 모른다.',
            '9': '분위기 파악을 잘하는 편이다.',
            '10': '쉽게 불안해진다.',
            '11': '하고 싶은 것이 내 뜻대로 되지 않는 경우 심하게 짜증이 난다.',
            '12': '첫 만남에서 자기소개하는 것이 어렵다.',
            '13': '여러 가지 일을 힘들이지 않고 해내곤 한다.',
            '14': '방해를 받더라도 내가 하던 일을 끝까지 해내는 편이다.',
            '15': '대부분의 일상이 즐겁다.',
            '16': '새로운 상황에 적응하는 것이 쉽지 않다.',
            '17': '실망하거나 실패하면 다른 일을 할 수 없을 정도로 기분이 나빠진다.',
            '18': '하루 중 대부분의 시간을 기분 좋게 보낸다.',
            '19': '바쁘게 생활하는 것이 즐겁다.',
            '20': '하기 싫은 일이라도 해야 한다면 끝까지 완수한다.',
            '21': '다른 사람의 입장이나 관점을 중요하게 여긴다.',
            '22': '새로운 사람들을 만나는 모임에 가는 것이 불편하다.',
            '23': '익숙하지 않은 모임에서 말을 하기보다는 지켜보는 편이다.',
            '24': '해야 할 일들은 미루지 않고 해내는 편이다.',
            '25': '하루에 3~4가지 이상의 일정을 잘 소화한다.',
            '26': '인생은 재미있는 일로 가득 차 있다.',
            '27': '직장 상사나 윗사람에게 싫은 소리를 들으면 화가 나서 아무것도 하고 싶지 않다.',
            '28': '계획(자신 또는 타인과의 약속)을 세우면 잘 지키는 편이다.',
            '29': '타인의 기분을 상하게 하지 않으려고 노력한다.',
            '30': '나는 행복하다.',
            '31': '눈치가 빠른 편이다.',
            '32': '과제나 일에 대한 집중력이 좋다.',
            '33': '아무것도 안 하고 가만히 있는 것은 시간 낭비라고 생각한다.',
            '34': '쉽게 우울해진다.',
            '35': '새로운 취미활동을 시도하는 것이 쉽지 않다.',
            '36': '조금 힘든 일이라도 나에게 의미가 있다면 끝까지 해낸다.',
            '37': '상대방의 기분에 대해 신경을 쓰는 편이다.',
            '38': '다른 사람들은 자주 나에게 좋은 일이 있냐고 묻는다.',
            '39': '어떤 문제를 해결하기 전까지 포기하지 않는 편이다.',
            '40': '새로운 시도가 부담스럽다(장소, 활동, 상황 등).',
            '41': '집에 있는 것보다 밖에 나가는 것을 좋아한다.',
            '42': '짜증이 나면 그 기분이 오래 지속된다.',
            '43': '말이나 행동을 하기 전에 주변 사람들이 어떤 반응을 보일지 생각해 본다.',
        },
        'meta_json': {'name': 'STS 6요인 기질검사 성인용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
# insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'

# insert 후 데이터 수정하고 싶을 때
# 기존 데이터를 덮어쓰도록 ON CONFLICT 절 추가 (sqlite)
conflict_columns = ['test_id', 'sub_test_json'] # pk 컬럼
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) ON CONFLICT({conflict_sql}) DO UPDATE SET {update_sql}'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건
